# Notebook 10 — Model Math Explainer

This notebook documents the full prediction pipeline in plain English:
- How delta_NRtg is computed and what values mean
- How the three base models produce probabilities
- How the meta-learner combines them (with actual weights from the trained model)
- Why 100% of 2026 predictions used the MC fallback instead of the ensemble
- MC logistic formula vs LightGBM series length model (accuracy comparison)
- Pairwise 2026 probabilities (all 16-team matchup pairs)

**Intended audience:** Someone familiar with the project but not with stacking ensembles or isotonic calibration.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from nba_predictor.models.ensemble import StackingEnsemble

# Load data
df = pd.read_parquet('../data/processed/series_dataset.parquet')
bracket_input = pd.read_csv('../data/predictions/2026/bracket_input.csv')
bracket_output = pd.read_csv('../data/predictions/2026/bracket_output.csv')
champ_probs = pd.read_csv('../data/predictions/2026/champion_probabilities.csv')

# Load trained ensemble
with open('../models/trained/ensemble_winner.pkl', 'rb') as f:
    ens = pickle.load(f)

print(f'Training series: {len(df)}')
print(f'Seasons: {df["season"].min()}–{df["season"].max()}')
print(f'2026 matchups in bracket_input: {len(bracket_input)}')

---
## 1. What is delta_NRtg?

**Net Rating (NRtg)** = points scored per 100 possessions minus points allowed per 100 possessions.  
A team with NRtg = +5 outscores opponents by 5 points per 100 possessions on average.

**Era normalization:** NRtg is z-scored within each season before being used as a feature.  
This is necessary because league-wide scoring has changed dramatically — a +5 NRtg in 1994 meant something different than in 2024.  
After normalization, a value of +1.0 means "one standard deviation above league average for that season."

**delta_NRtg = higher_seed_NRtg_norm − lower_seed_NRtg_norm**  
- Positive: higher seed was the better team by net rating  
- Negative: lower seed was actually the better team (model sees upset risk)  
- |delta| < 0.5: essentially a coin flip  
- |delta| > 1.5: strong favorite, but upsets still happen

In [ ]:
# Show real examples from recent seasons
examples = [
    (2025, 'OKC vs IND (Finals)', 1.76, 7, 'Higher seed (OKC) won'),
    (2025, 'BOS vs ORL (R1)', 1.61, 5, 'Higher seed (BOS) won'),
    (2025, 'CLE vs MIA (R1)', 1.48, 4, 'Higher seed (CLE) won — sweep'),
    (2025, 'OKC vs DEN (R2)', 1.48, 7, 'Higher seed (OKC) won — went 7'),
    (2025, 'HOU vs GSW (R1)', 0.20, 7, 'UPSET — lower seed (GSW) won'),
    (2025, 'DEN vs LAC (R2)', -0.15, 7, 'Higher seed (DEN) won despite neg delta'),
    (2025, 'LAL vs MIN (R1)', -0.65, 5, 'UPSET — lower seed (MIN) won'),
    (2025, 'IND vs MIL (R1)', -0.05, 5, 'UPSET — lower seed (IND) won'),
]

ex_df = pd.DataFrame(examples, columns=['Season', 'Matchup', 'delta_NRtg', 'Series Length', 'Outcome'])
ex_df = ex_df.sort_values('delta_NRtg', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if 'won' in r and 'UPSET' not in r else 'red' for r in ex_df['Outcome']]
bars = ax.barh(ex_df['Matchup'], ex_df['delta_NRtg'], color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=1)
ax.axvline(0.5, color='gray', linewidth=0.8, linestyle='--', label='|delta| < 0.5 = coin flip zone')
ax.axvline(-0.5, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('delta_NRtg (higher seed − lower seed, era-normalized standard deviations)')
ax.set_title('delta_NRtg vs Actual 2025 Series Outcomes')
green_patch = mpatches.Patch(color='green', label='Higher seed won (expected)')
red_patch = mpatches.Patch(color='red', label='Lower seed won (upset)')
ax.legend(handles=[green_patch, red_patch, plt.Line2D([0],[0],color='gray',linestyle='--',label='Coin-flip zone (±0.5)')])
plt.tight_layout()
plt.show()

print(ex_df.to_string(index=False))

---
## 2. Three Base Models

Each base model independently estimates P(higher_seed_wins_series) from the same 42 features.

| Model | Input Preprocessing | Core Behavior |
|-------|--------------------|--------------|
| **Logistic Regression** | z-scored (StandardScaler stored as `scaler_base`) | Linear combination of all features — clean linear signal from delta_NRtg, seed_diff |
| **XGBoost** | Raw features (no scaling needed) | Decision tree ensemble; captures non-linear thresholds (e.g., delta_NRtg > 1.5 qualitatively different from 0.5) |
| **LightGBM** | Raw features | Leaf-wise tree growth; slightly different non-linear splits |

**Why three models?** They make different errors. LR may miss a threshold interaction ("dominant star + big NRtg gap → nearly guaranteed win"); XGB may overfit to small-sample patterns. Averaging their predictions reduces variance.

In [ ]:
# Get base model probabilities for 2026 bracket input
feats = ens.feature_cols
X = bracket_input[feats].fillna(0)

base_probs = {}
for name, (mtype, model) in ens._base_models.items():
    if mtype == 'scaled':
        Xs = ens.scaler_base.transform(X)
        probs = model.predict_proba(Xs)[:, 1]
    else:
        probs = model.predict_proba(X.values)[:, 1]
    base_probs[name] = probs

matchups = bracket_input['matchup_id'].tolist() if 'matchup_id' in bracket_input.columns else [
    f"{row['higher_seed']} vs {row['lower_seed']}" for _, row in bracket_input.iterrows()
]

bp_df = pd.DataFrame(base_probs, index=matchups)
bp_df['MC_fallback'] = bracket_output[bracket_output['round'] == 'first_round']['p_higher_seed_wins'].values
print('=== Base Model Probabilities (2026 First Round) ===')
print('Values > 0.70 are SATURATED (likely to map to ~1.0 after meta-learner)')
print()
print(bp_df.round(3).to_string())

---
## 3. How the Meta-Learner Combines the Three Models

### Step 1: Out-of-Fold (OOF) Predictions

The meta-learner cannot be trained on the same predictions it will be scored on.  
Instead, **out-of-fold predictions** are generated:

```
For each walk-forward fold (train on seasons < N, test on season N):
    1. Fit LR + XGB + LightGBM on training seasons
    2. Predict on season N (never seen by these partial models)
    3. Record 3 probabilities per series

After all folds: every historical series has 3 OOF probabilities
    oof_matrix = shape (345 series, 3 cols: p_lr, p_xgb, p_lgbm)
```

### Step 2: Meta-Learner Training

A Logistic Regression (`C=0.5`) is trained on `oof_matrix → actual_wins`.  
Then `CalibratedClassifierCV(cv=3)` wraps it in 3 inner calibration folds.

### Step 3: Actual Weights (from trained model)

The 3 calibration folds each produce a LogisticRegression with slightly different weights:

In [ ]:
# Extract and display actual meta-learner weights
cal = ens.meta_calibrated
model_names = ['LogReg', 'XGBoost', 'LightGBM']

fold_weights = []
for i, cc in enumerate(cal.calibrated_classifiers_):
    lr = cc.estimator
    fold_weights.append(lr.coef_.flatten())

fold_weights = np.array(fold_weights)
avg_weights = fold_weights.mean(axis=0)

weights_df = pd.DataFrame(fold_weights, columns=model_names, index=[f'Cal fold {i}' for i in range(3)])
weights_df.loc['Average'] = avg_weights
print('Meta-Learner Coefficients (how much each base model is trusted):')
print(weights_df.round(3).to_string())
print()
print('Interpretation:')
print(f'  LogReg  weight ≈ {avg_weights[0]:.2f} — dominates because NRtg/seed_diff are largely linear')
print(f'  XGBoost weight ≈ {avg_weights[1]:.2f} — adds non-linear threshold interactions')
print(f'  LightGBM weight ≈ {avg_weights[2]:.2f} — adds different non-linear splits')

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
width = 0.22
for i in range(3):
    ax.bar(x + i*width, fold_weights[i], width, label=f'Cal fold {i}', alpha=0.8)
ax.bar(x + 3*width, avg_weights, width, label='Average', color='black', alpha=0.9)
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(model_names)
ax.set_ylabel('Meta-learner coefficient (weight)')
ax.set_title('How Much the Meta-Learner Trusts Each Base Model')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Why ALL 2026 Predictions Used the MC Fallback (Critical Issue)

### The Problem: Calibration Saturation

The base models are **heavily overfit** to the 345 training samples.  
With 300-estimator trees and 42 features on only 345 rows, the models essentially memorize training outcomes.

This creates a mismatch:

```
OOF predictions (used to train meta-learner calibration):
    Models trained on SUBSETS → moderate probabilities: 0.40–0.85

Full-data predictions (used at inference time):
    Models trained on ALL 345 rows → saturated: 0.55–1.00
```

The isotonic calibrator was fitted on OOF probabilities (0.4–0.85 range).  
When full-data base models output 0.88–1.0, the calibrator **cannot extrapolate** — it maps everything to 1.0.

When `predict_proba()` returns a probability ≥ 1.0 (or causes a numerical issue),  
the bracket simulator silently catches the exception and falls back to the MC NRtg formula.  
**This happened for 100% of predictions — 120 team pairs × 16 teams = all matchups.**

In [ ]:
# Demonstrate the calibration saturation
def mc_fallback_prob(delta_nrtg, home_court_boost=2.5):
    """System B: NRtg-only MC logistic formula used for all 2026 predictions."""
    return 1 / (1 + np.exp(-0.104 * (delta_nrtg + home_court_boost)))

# Get base probs for training data to show OOF vs full-data distribution
X_train = df[feats].fillna(0)
train_base_probs = {}
for name, (mtype, model) in ens._base_models.items():
    if mtype == 'scaled':
        Xs = ens.scaler_base.transform(X_train)
        train_base_probs[name] = model.predict_proba(Xs)[:, 1]
    else:
        train_base_probs[name] = model.predict_proba(X_train.values)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, name in zip(axes, ['lr', 'xgb', 'lgbm']):
    ax.hist(train_base_probs[name], bins=30, alpha=0.6, color='steelblue', label='Training (full-data, overfit)')
    ax.axvline(0.82, color='red', linewidth=2, label='2026 inputs start here →')
    ax.set_title(f'{name.upper()} probability distribution')
    ax.set_xlabel('P(higher seed wins)')
    ax.legend(fontsize=8)

plt.suptitle('Base Model Output Distribution on Training Data (Memorized)\n'
             'Red line = minimum 2026 input probability → calibrator maps everything to 1.0')
plt.tight_layout()
plt.show()

print('Training data base model stats (full-data = overfit):')
for name, probs in train_base_probs.items():
    print(f'  {name}: min={probs.min():.3f}  max={probs.max():.3f}  mean={probs.mean():.3f}')

In [ ]:
# Show the before/after: ensemble (broken) vs MC fallback (actual 2026 output)
r1_matchups = bracket_output[bracket_output['round'] == 'first_round'].copy()

comparison = pd.DataFrame({
    'Matchup': [f"{r['higher_seed']} vs {r['lower_seed']}" for _, r in r1_matchups.iterrows()],
    'Ensemble_LR': base_probs['lr'],
    'Ensemble_XGB': base_probs['xgb'],
    'Ensemble_LGBM': base_probs['lgbm'],
    'MC_Fallback_Used': r1_matchups['p_higher_seed_wins'].values,
})
comparison['delta_NRtg'] = bracket_input['delta_NRtg'].values

print('=== 2026 R1: Ensemble (broken) vs MC Fallback (actual output) ===')
print('Note: All Ensemble values are saturated (0.55-1.0) → fallback triggered')
print(comparison.round(3).to_string(index=False))

print('\n⚠️  WARNING: 0% of 2026 predictions used the stacking ensemble.')
print('All 120 pairwise probabilities came from the NRtg-only MC formula.')
print('Player quality, injury flags, and H2H features were IGNORED.')
print('Fix: retrain with reduced model complexity + sigmoid calibration (Phase 2).')

---
## 5. The MC Fallback Formula

**System B: NRtg-only logistic formula**

```
p_home_wins = 1 / (1 + exp(−0.104 × (delta_NRtg + 2.5)))
p_away_wins = 1 / (1 + exp(−0.104 × (delta_NRtg − 2.5)))
```

The `0.104` coefficient converts NRtg standard deviations into log-odds.  
The `±2.5` is the **home court boost** — playing at home is worth ~2.5 NRtg points historically.

In the 2-2-1-1-1 playoff format: games 1, 2, 5 are at the higher seed's arena (home), games 3, 4, 6 at lower seed (away), game 7 at higher seed (home).

Series probability is computed by simulating the 2-2-1-1-1 format game by game using `p_home` and `p_away` depending on venue.

In [ ]:
# Illustrate the home court boost effect on the MC formula
delta_range = np.linspace(-3, 3, 200)
p_home = 1 / (1 + np.exp(-0.104 * (delta_range + 2.5)))
p_away = 1 / (1 + np.exp(-0.104 * (delta_range - 2.5)))
p_neutral = 1 / (1 + np.exp(-0.104 * delta_range))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(delta_range, p_home, label='At home (boost=+2.5)', color='green', linewidth=2)
ax.plot(delta_range, p_neutral, label='Neutral court', color='gray', linewidth=2, linestyle='--')
ax.plot(delta_range, p_away, label='Away (boost=−2.5)', color='red', linewidth=2)
ax.axhline(0.5, color='black', linewidth=0.8, linestyle=':')
ax.axvline(0, color='black', linewidth=0.8, linestyle=':')

# Annotate 2026 R1 matchups
r1_deltas = bracket_input['delta_NRtg'].values
r1_labels = comparison['Matchup'].values
for d, lbl in zip(r1_deltas[:4], r1_labels[:4]):  # East R1
    y = 1 / (1 + np.exp(-0.104 * (d + 2.5)))
    ax.annotate(lbl.split(' vs ')[0], (d, y), textcoords='offset points', xytext=(0, 8), fontsize=7, ha='center')

ax.set_xlabel('delta_NRtg (higher_seed − lower_seed, era-normalized s.d.)')
ax.set_ylabel('P(higher seed wins game)')
ax.set_title('MC Fallback: Home Court Boost Effect on Per-Game Win Probability')
ax.legend()
plt.tight_layout()
plt.show()

print('Home court boost examples:')
for d in [0.0, 0.5, 1.0, 1.5]:
    p_h = 1 / (1 + np.exp(-0.104 * (d + 2.5)))
    p_a = 1 / (1 + np.exp(-0.104 * (d - 2.5)))
    print(f'  delta_NRtg={d:+.1f}: P(win at home)={p_h:.3f}  P(win away)={p_a:.3f}')

---
## 6. Series Length: LightGBM vs MC — Accuracy Comparison

**Series length uses a separate LightGBM multiclass model** (predicts 4, 5, 6, or 7 games).  
MC game-by-game simulation is the fallback: it samples individual game outcomes using the per-game win probability and counts how many games it takes to reach 4 wins.

**Why MC struggles with series length:**  
If P(win game) = 0.54 for the higher seed, random simulation almost always produces 6 or 7 games — it never "knows" that a dominant star or massive roster gap can produce a sweep.
MC only uses delta_NRtg. LightGBM uses all 42+ features including seed_diff, delta_VORP, delta_Star_BPM.

In [ ]:
# Load LightGBM length model and compare to MC on last 20 series
import math

with open('../models/trained/lgbm_length.pkl', 'rb') as f:
    lgbm_length = pickle.load(f)

# Get feature names that the length model uses
lgbm_feat_names = lgbm_length.feature_name_
available = [c for c in lgbm_feat_names if c in df.columns]
print(f'Length model features: {len(lgbm_feat_names)} expected, {len(available)} available in dataset')

# Predict on last 20 series
recent = df.tail(20).copy()
X_recent = recent[available].fillna(0)
if len(available) < len(lgbm_feat_names):
    # Pad missing features with 0
    X_padded = pd.DataFrame(0, index=X_recent.index, columns=lgbm_feat_names)
    X_padded[available] = X_recent[available]
    X_recent = X_padded

lgbm_preds = lgbm_length.predict(X_recent)
lgbm_classes = lgbm_length.classes_

# MC simulation for same series
def mc_series_length_modal(delta_nrtg, home_boost=2.5, n_sim=10000):
    game_locs = ['H','H','A','A','H','A','H']
    lengths = []
    for _ in range(n_sim):
        wins_h, wins_a = 0, 0
        for g, loc in enumerate(game_locs):
            boost = home_boost if loc == 'H' else -home_boost
            p = 1 / (1 + math.exp(-0.104 * (delta_nrtg + boost)))
            if np.random.random() < p:
                wins_h += 1
            else:
                wins_a += 1
            if wins_h == 4 or wins_a == 4:
                lengths.append(g + 1)
                break
    counts = pd.Series(lengths).value_counts()
    return counts.idxmax()

np.random.seed(42)
mc_preds = [mc_series_length_modal(row['delta_NRtg']) for _, row in recent.iterrows()]

results = pd.DataFrame({
    'Season': recent['season'].values,
    'delta_NRtg': recent['delta_NRtg'].values.round(2),
    'Actual_Length': recent['series_length'].values,
    'LightGBM_Pred': lgbm_preds,
    'MC_Modal_Pred': mc_preds,
})
results['LGBM_Correct'] = (results['LightGBM_Pred'] == results['Actual_Length'])
results['MC_Correct'] = (results['MC_Modal_Pred'] == results['Actual_Length'])

lgbm_acc = results['LGBM_Correct'].mean()
mc_acc = results['MC_Correct'].mean()
print(f'\nLast 20 series — exact length prediction accuracy:')
print(f'  LightGBM: {results["LGBM_Correct"].sum()}/20 = {lgbm_acc:.0%}')
print(f'  MC modal: {results["MC_Correct"].sum()}/20 = {mc_acc:.0%}')
print()
print(results.to_string(index=False))

In [ ]:
# Visualize MC vs LightGBM predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (pred_col, title, acc) in zip(axes, [
    ('LightGBM_Pred', f'LightGBM ({lgbm_acc:.0%} exact)', lgbm_acc),
    ('MC_Modal_Pred', f'MC Simulation ({mc_acc:.0%} exact)', mc_acc),
]):
    x = results['Actual_Length'].values
    y = results[pred_col].values
    correct = x == y
    ax.scatter(x[correct], y[correct], c='green', s=80, label='Correct', zorder=3, alpha=0.9)
    ax.scatter(x[~correct], y[~correct], c='red', s=80, label='Wrong', zorder=3, alpha=0.6, marker='x')
    ax.plot([4, 7], [4, 7], 'k--', alpha=0.4, label='Perfect')
    ax.set_xlim(3.5, 7.5); ax.set_ylim(3.5, 7.5)
    ax.set_xticks([4, 5, 6, 7]); ax.set_yticks([4, 5, 6, 7])
    ax.set_xlabel('Actual Series Length'); ax.set_ylabel('Predicted Series Length')
    ax.set_title(title); ax.legend()

plt.suptitle('Series Length Model Comparison (Last 20 Training Series)')
plt.tight_layout()
plt.show()

# Key failure modes
print('Key failure mode — sweeps (4-game series):')
sweeps = results[results['Actual_Length'] == 4]
print(f'  Actual sweeps: {len(sweeps)}')
print(f'  LightGBM predicted sweep: {(sweeps["LightGBM_Pred"] == 4).sum()}')
print(f'  MC predicted sweep: {(sweeps["MC_Modal_Pred"] == 4).sum()}')
print('  MC defaults to 7-game predictions when win probability is near 50%')

---
## 7. All Pairwise 2026 Probabilities

The bracket simulator pre-computes P(higher_seed_wins) for all C(16,2) = 120 team pairs before running the bracket.  
This allows Round 2+ matchups to be predicted without re-running inference.

**These values are from the MC fallback (NRtg-only formula).**  
After Phase 2 retrain, they should be regenerated from the ensemble.

Note: The simulator is rerun here to extract the probability table.

In [ ]:
# Load pairwise probabilities if they exist, otherwise note they need to be generated
import os

pairwise_path = '../data/predictions/2026/pairwise_probabilities.csv'
if os.path.exists(pairwise_path):
    pair_df = pd.read_csv(pairwise_path)
    print(f'Loaded pairwise probabilities: {len(pair_df)} pairs')
else:
    print('pairwise_probabilities.csv not yet generated.')
    print('It will be created automatically on the next bracket_simulator run (Phase 1 task).')
    print('Showing bracket_output.csv R1 probabilities instead:')
    r1 = bracket_output[bracket_output['round'] == 'first_round'][['higher_seed','lower_seed','p_higher_seed_wins']]
    r1 = r1.rename(columns={'p_higher_seed_wins': 'p_higher_wins'})
    r1['p_lower_wins'] = 1 - r1['p_higher_wins']
    print(r1.round(3).to_string(index=False))

In [ ]:
# Champion probabilities
print('2026 Championship Odds (from 10,000 Monte Carlo simulations, MC fallback probabilities):')
print()
champ_sorted = champ_probs.sort_values('p_champion', ascending=False)
print(champ_sorted.to_string(index=False))

# Bar chart
top10 = champ_sorted.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(top10['team'], top10['p_champion'] * 100, alpha=0.85,
              color=['gold' if t == champ_sorted.iloc[0]['team'] else 'steelblue' for t in top10['team']])
ax.set_ylabel('Championship probability (%)')
ax.set_title('2026 NBA Championship Odds (Top 10)\nSource: MC NRtg-only fallback — ensemble not used')
for bar, val in zip(bars, top10['p_champion']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f'{val*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\n⚠️ These odds reflect NRtg gap only. Player quality (BPM, VORP), injuries,')
print('and head-to-head records are not incorporated until Phase 2 retrain.')

---
## 8. Summary: Known Issues and Phase 2 Fix Plan

| Issue | Root Cause | Impact | Phase 2 Fix |
|-------|-----------|--------|-------------|
| **Ensemble not used (100% MC fallback)** | Base models overfit (300 trees, 42 features, 345 samples). Isotonic calibrator can't extrapolate. | All predictions NRtg-only. Player/injury/H2H features ignored. | Reduce complexity: LR C→0.05, XGB depth=2/trees=50, LGBM leaves=7/trees=50. Switch to sigmoid calibration. |
| **H2H features always null (all 345 training rows)** | SEASON_ID format mismatch in game log join (e.g., '22025' vs 2025 int) | H2H feature never learned. Model behaves as if H2H=0 always. | Fix `_compute_h2h()` join, re-process dataset, retrain. |
| **Length model uses 43 features, dataset has 49** | Model trained before 6 new features were added | 6 features unused (gracefully handled via reindex/fillna) | Retrain length model after main retrain with same feature set. |
| **No fallback transparency** | Silent `logger.debug` catch in `_model_prob()` | No way to know predictions were MC-based without debugging | Add `logger.WARNING`, track fallback_log, add `prediction_source` column to CSVs. |
| **Pairwise probability table not saved** | Not written in main entry point | Can't review hypothetical matchup odds | 3-line fix in bracket_simulator.py main (Phase 1 task). |

### Current Predictions Are Still Useful

The MC fallback produces **internally consistent** probabilities based on net rating differential.  
NRtg is the single strongest predictor of series outcomes. The predictions are defensible as a NRtg-based baseline.  
They simply don't use the additional signal that the ensemble was built to capture.

### What Phase 2 Changes

After the retrain:
- OKC's 12% championship odds may shift significantly (they currently reflect only their regular-season NRtg advantage)
- Teams with strong stars or favorable H2H matchups will see adjusted odds
- Injury-adjusted VORP will properly downweight teams missing key players
- All of this comes with explicit `prediction_source` tracking so you always know which system produced each number